# Introduction to Linear Model

### Student Performance (Multiple Linear Regression)

Exploring Factors Affecting Student Performance

https://www.kaggle.com/datasets/nikhil7280/student-performance-multiple-linear-regression

In [3]:
import numpy as np
import pandas as pd
from ydata_profiling import ProfileReport

import warnings
from tqdm import tqdm

# Suppress warnings
warnings.filterwarnings("ignore")


Simplified EDA

In [4]:
df = pd.read_csv('../data/student-perf/Student_Performance.csv')

In [5]:
report = ProfileReport(
    df,
    title="Student Performance Report",
    explorative=True,
    progress_bar=False)
report.to_file("../reports/student_performance_report.html")

100%|██████████| 6/6 [00:00<00:00, 873.39it/s]


In [13]:
from sklearn.preprocessing import LabelEncoder

# Initialize the label encoder
label_encoder = LabelEncoder()

# Fit and transform the 'Extracurricular Activities' column
df['Extracurricular Activities'] = label_encoder.fit_transform(df['Extracurricular Activities'])

# Display the first few rows to verify
df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,1,9,1,91.0
1,4,82,0,4,2,65.0
2,8,51,1,7,2,45.0
3,5,52,1,5,2,36.0
4,7,75,0,8,5,66.0


#### Linear Models

In [23]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, SGDRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier


feature_columns = [col for col in df.columns if col not in ['Performance Index']]

# Prepare the data
X = df[feature_columns].values  # Independent variable
y = df['Performance Index'].values  # Dependent variable

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize models
models = {
    "LinearRegressionOLS": LinearRegression(),
    "LinearRegressionSGD": SGDRegressor(max_iter=1000, random_state=42, learning_rate='adaptive', eta0=0.1),
    "LinearRegressionRidge": Ridge(alpha=1.0),
    "LinearRegressionLasso": Lasso(alpha=0.1),
    "LinearRegressionElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "NaiveBayes": GaussianNB(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "DecisionTree": DecisionTreeClassifier(max_depth=20,random_state=42)
}

# Train and evaluate each model
mse_results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mse_results[name] = mse

# Print train and test MSE for each model
print("\nTrain and Test MSE for each model:")
for name, mse in mse_results.items():
    train_mse = mean_squared_error(y_train, models[name].predict(X_train))
    test_mse = mse
    print(f"{name}: Train MSE = {train_mse:.2f}, Test MSE = {test_mse:.2f}")


Train and Test MSE for each model:
LinearRegressionOLS: Train MSE = 4.17, Test MSE = 4.08
LinearRegressionSGD: Train MSE = 4.32, Test MSE = 4.14
LinearRegressionRidge: Train MSE = 4.17, Test MSE = 4.08
LinearRegressionLasso: Train MSE = 4.22, Test MSE = 4.14
LinearRegressionElasticNet: Train MSE = 4.20, Test MSE = 4.12
NaiveBayes: Train MSE = 39.41, Test MSE = 37.30
KNN: Train MSE = 10.49, Test MSE = 13.28
DecisionTree: Train MSE = 0.52, Test MSE = 8.81


#### Ensemble based models

In [28]:
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
# Initialize the models
additional_models = {
    "SVM": SVR(kernel='rbf', C=1.0, epsilon=0.1),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "CatBoost": CatBoostRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbose=0),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42)
}

# Train and evaluate each model
for name, model in additional_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mse_results[name] = mse

# Print train and test MSE for each model
print("\nTrain and Test MSE for additional models:")
for name, model in additional_models.items():
    train_mse = mean_squared_error(y_train, model.predict(X_train))
    test_mse = mse_results[name]
    print(f"{name}: Train MSE = {train_mse:.2f}, Test MSE = {test_mse:.2f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000063 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 90
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 5
[LightGBM] [Info] Start training from score 55.311500

Train and Test MSE for additional models:
SVM: Train MSE = 5.41, Test MSE = 5.39
XGBoost: Train MSE = 3.33, Test MSE = 4.33
LightGBM: Train MSE = 3.65, Test MSE = 4.31
CatBoost: Train MSE = 4.14, Test MSE = 4.35
RandomForest: Train MSE = 0.94, Test MSE = 5.16


#### Neural Networks

In [39]:
import torch
from sklearn.preprocessing import StandardScaler

import torch.nn as nn
import torch.optim as optim

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Define the MLP model
class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x

# Initialize the model, loss function, and optimizer
input_size = X_train.shape[1]
model = MLP(input_size)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model
epochs = 1000
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    y_pred = model(X_train_tensor)
    loss = criterion(y_pred, y_train_tensor)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")

# Evaluate the model
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_tensor)
    test_loss = criterion(y_test_pred, y_test_tensor)
    print(f"Test MSE: {test_loss.item():.4f}")

Epoch 100/1000, Loss: 1930.5172
Epoch 200/1000, Loss: 90.5274
Epoch 300/1000, Loss: 52.4664
Epoch 400/1000, Loss: 42.8137
Epoch 500/1000, Loss: 35.5108
Epoch 600/1000, Loss: 29.0945
Epoch 700/1000, Loss: 23.5786
Epoch 800/1000, Loss: 19.0498
Epoch 900/1000, Loss: 15.4358
Epoch 1000/1000, Loss: 12.5401
Test MSE: 12.5687


In [40]:
import torch
from sklearn.preprocessing import StandardScaler

import torch.nn as nn
import torch.optim as optim

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Reshape input for LSTM (batch_size, sequence_length, input_size)
X_train_tensor = X_train_tensor.unsqueeze(1)
X_test_tensor = X_test_tensor.unsqueeze(1)

# Define the LSTM model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # Take the output of the last time step
        return out

# Initialize the model, loss function, and optimizer
input_size = X_train.shape[1]
hidden_size = 64
num_layers = 2
model = LSTMModel(input_size, hidden_size, num_layers)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model
epochs = 1000
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    y_pred = model(X_train_tensor)
    loss = criterion(y_pred, y_train_tensor)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")

# Evaluate the model
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_tensor)
    test_loss = criterion(y_test_pred, y_test_tensor)
    print(f"Test MSE: {test_loss.item():.4f}")

Epoch 100/1000, Loss: 2905.7173
Epoch 200/1000, Loss: 1926.2614
Epoch 300/1000, Loss: 1470.3459
Epoch 400/1000, Loss: 1163.2555
Epoch 500/1000, Loss: 937.1789
Epoch 600/1000, Loss: 759.9985
Epoch 700/1000, Loss: 613.5988
Epoch 800/1000, Loss: 501.2657
Epoch 900/1000, Loss: 412.1020
Epoch 1000/1000, Loss: 339.8988
Test MSE: 329.1992


In [41]:
import torch
from sklearn.preprocessing import StandardScaler

import torch.nn as nn
import torch.optim as optim

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Define the Transformer model
class TransformerModel(nn.Module):
    def __init__(self, input_size, d_model, nhead, num_layers, dim_feedforward, dropout=0.1):
        super(TransformerModel, self).__init__()
        self.input_layer = nn.Linear(input_size, d_model)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.output_layer = nn.Linear(d_model, 1)

    def forward(self, src):
        src = self.input_layer(src)
        src = self.transformer(src, src)
        output = self.output_layer(src[:, -1, :])  # Use the last time step
        return output

# Initialize the model, loss function, and optimizer
input_size = X_train.shape[1]
d_model = 64
nhead = 4
num_layers = 2
dim_feedforward = 128
model = TransformerModel(input_size, d_model, nhead, num_layers, dim_feedforward)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model
epochs = 1000
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    y_pred = model(X_train_tensor.unsqueeze(1))  # Add sequence dimension
    loss = criterion(y_pred, y_train_tensor)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")

# Evaluate the model
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_tensor.unsqueeze(1))  # Add sequence dimension
    test_loss = criterion(y_test_pred, y_test_tensor)
    print(f"Test MSE: {test_loss.item():.4f}")

Epoch 100/1000, Loss: 2184.0544
Epoch 200/1000, Loss: 1278.3195
Epoch 300/1000, Loss: 653.5947
Epoch 400/1000, Loss: 282.1443
Epoch 500/1000, Loss: 126.9255
Epoch 600/1000, Loss: 62.7932
Epoch 700/1000, Loss: 35.0438
Epoch 800/1000, Loss: 21.9157
Epoch 900/1000, Loss: 15.2010
Epoch 1000/1000, Loss: 11.5821
Test MSE: 10.3565


In [50]:
import optuna
from sklearn.model_selection import train_test_split
from xgboost import DMatrix, train as xgb_train

# Split the training data into a smaller training set and a validation set for early stopping
X_train_sub, X_val, y_train_sub, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Define the objective function for Optuna
def objective(trial):
    # Suggest hyperparameters
    max_depth = trial.suggest_int("max_depth", 3, 10)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3, log=True)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    reg_alpha = trial.suggest_float("reg_alpha", 0.0, 10.0)
    reg_lambda = trial.suggest_float("reg_lambda", 0.0, 10.0)

    # Prepare the data for XGBoost
    dtrain = DMatrix(X_train_sub, label=y_train_sub)
    dval = DMatrix(X_val, label=y_val)
    dtest = DMatrix(X_test)

    # Define the parameters
    params = {
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "reg_alpha": reg_alpha,
        "reg_lambda": reg_lambda,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "seed": 42
    }

    # Train the model with early stopping
    evals = [(dtrain, "train"), (dval, "eval")]
    model = xgb_train(params, dtrain, num_boost_round=10000, evals=evals, early_stopping_rounds=50, verbose_eval=False)

    # Predict on the test set
    y_pred = model.predict(dtest)

    # Calculate the mean squared error
    mse = mean_squared_error(y_test, y_pred)
    return mse

# Create a study and optimize
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.HyperbandPruner())
study.optimize(objective, n_trials=50)

# Print the best hyperparameters and the corresponding MSE
print("Best hyperparameters:", study.best_params)
print("Best MSE:", study.best_value)

[I 2025-05-25 19:03:11,143] A new study created in memory with name: no-name-4dd68d78-2abe-4535-b918-b6b237d980b7
[I 2025-05-25 19:03:11,336] Trial 0 finished with value: 4.501244594802438 and parameters: {'max_depth': 5, 'learning_rate': 0.2536999076681772, 'subsample': 0.8659969709057025, 'colsample_bytree': 0.7993292420985183, 'reg_alpha': 1.5601864044243652, 'reg_lambda': 1.5599452033620265}. Best is trial 0 with value: 4.501244594802438.
[I 2025-05-25 19:03:11,524] Trial 1 finished with value: 4.3050510008551885 and parameters: {'max_depth': 3, 'learning_rate': 0.19030368381735815, 'subsample': 0.8005575058716043, 'colsample_bytree': 0.8540362888980227, 'reg_alpha': 0.20584494295802447, 'reg_lambda': 9.699098521619943}. Best is trial 1 with value: 4.3050510008551885.
[I 2025-05-25 19:03:12,422] Trial 2 finished with value: 4.289789115950679 and parameters: {'max_depth': 9, 'learning_rate': 0.020589728197687916, 'subsample': 0.5909124836035503, 'colsample_bytree': 0.591702254926716

Best hyperparameters: {'max_depth': 3, 'learning_rate': 0.05323458539106301, 'subsample': 0.5007447210575431, 'colsample_bytree': 0.569987334762439, 'reg_alpha': 3.972792364171065, 'reg_lambda': 8.996667735732014}
Best MSE: 4.165988793348561


In [49]:
study.best_trial.number

26

In [48]:
from xgboost import DMatrix, train as xgb_train

# Prepare the data for XGBoost
dtrain = DMatrix(X_train, label=y_train)
dtest = DMatrix(X_test)

# Use the best hyperparameters from the Optuna study
best_params = study.best_params
best_params.update({
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "seed": 42
})

# Train the final model
final_model = xgb_train(best_params, dtrain, num_boost_round=study.best_trial.number)

# Predict on the test set
y_test_pred = final_model.predict(dtest)

# Evaluate the model
final_mse = mean_squared_error(y_test, y_test_pred)
print(f"Final Test MSE: {final_mse:.4f}")

Final Test MSE: 58.7476


### Testing INIT Score for LGBM Models


#### Binary Classification with a Bank Churn Dataset  
Playground Series - Season 4, Episode 1

https://www.kaggle.com/competitions/playground-series-s4e1/data

In [16]:

import pandas as pd
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")


In [17]:
df = pd.read_csv("../data/playground-series-s4e1/train.csv")

# Display the first few rows of the dataset
df.head()

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,0,15674932,Okwudilichukwu,668,France,Male,33.0,3,0.00,2,1.0,0.0,181449.97,0
1,1,15749177,Okwudiliolisa,627,France,Male,33.0,1,0.00,2,1.0,1.0,49503.50,0
2,2,15694510,Hsueh,678,France,Male,40.0,10,0.00,2,1.0,0.0,184866.69,0
3,3,15741417,Kao,581,France,Male,34.0,2,148882.54,1,1.0,1.0,84560.88,0
4,4,15766172,Chiemenam,716,Spain,Male,33.0,5,0.00,2,1.0,1.0,15068.83,0


In [18]:
df.shape

(165034, 14)

In [19]:
df.Exited.value_counts()

Exited
0    130113
1     34921
Name: count, dtype: int64

In [33]:
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Prepare the data
X = df.drop(columns=['Exited', 'id', 'CustomerId', 'Surname'])
y = df['Exited']

# Convert text columns to 'category' dtype
for col in ['Geography', 'Gender']:
    X[col] = X[col].astype('category')

# Set 'Geography' and 'Gender' as categorical features
categorical_features = ['Geography', 'Gender']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize the LightGBM classifier with categorical features
model = LGBMClassifier(
    n_estimators=4,
    boosting_type='gbdt',
    learning_rate=0.1,
    categorical_feature=categorical_features,
    force_row_wise=True,
    random_state=42,
    verbose=-1
)

# Train the model
model.fit(X_train, y_train, categorical_feature=categorical_features)
print
# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"4 trees Model Accuracy: {accuracy:.4f}")

# Get predicted probabilities for the positive class
y_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)
print(f"4 trees Model AUC: {auc:.4f}")


4 trees Model Accuracy: 0.7884
4 trees Model AUC: 0.8808


In [34]:

# Get the predicted probabilities for the positive class from the first model
init_score = model.predict(X_train, raw_score=True)

# Train a second LGBM model using the init_score
second_model = LGBMClassifier(
    n_estimators=2,
    learning_rate=0.1,
    categorical_feature=categorical_features,
    random_state=42,
    verbose=-1
)

# Pass init_score to fit method
second_model.fit(
    X_train,
    y_train,
    init_score=init_score
)

# Evaluate the second model
y_pred_second = second_model.predict(X_test)
accuracy_second = accuracy_score(y_test, y_pred_second)
print(f"Second +2 Trees Model Accuracy: {accuracy_second:.4f}")

# AUC for the second model
y_proba_second = second_model.predict_proba(X_test)[:, 1]
auc_second = roc_auc_score(y_test, y_proba_second)
print(f"Second +2 Trees Model AUC: {auc_second:.4f}")

Second +2 Trees Model Accuracy: 0.8142
Second +2 Trees Model AUC: 0.8793


In [32]:

# Initialize the LightGBM modelegressor
model_6trees = LGBMClassifier(
    n_estimators=6,
    learning_rate=0.1,
    categorical_feature=categorical_features,
    random_state=42,
    verbose=-1
)

# Train the model
model_6trees.fit(X_train, y_train)

# Predict on the test set
y_pred = model_6trees.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"6 Trees Model Accuracy: {accuracy:.4f}")

# Get predicted probabilities for the positive class
y_proba = model_6trees.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)
print(f"6 Trees Model AUC: {auc:.4f}")

6 Trees Model Accuracy: 0.8346
6 Trees Model AUC: 0.8819


In [31]:
from scipy.special import expit  # Sigmoid function

# 4. For prediction: sum raw scores from both models, then apply sigmoid
raw1 = model.predict(X_test, raw_score=True)
raw2 = second_model.predict(X_test, raw_score=True)
final_raw = raw1 + raw2
final_proba = expit(final_raw)  # This is the final predict_proba for the positive
final_pred = (final_proba >= 0.5).astype(int)

# Evaluate the combined model
combined_accuracy = accuracy_score(y_test, final_pred)
print(f"Combined Model Accuracy: {combined_accuracy:.4f}")

# AUC for the combined model
combined_auc = roc_auc_score(y_test, final_proba)
print(f"Combined Model AUC: {combined_auc:.4f}")

Combined Model Accuracy: 0.8346
Combined Model AUC: 0.8819
